In [91]:
%pip install pandas transformers deepparse huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [92]:
import torch
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
    torch.set_default_device('cuda')
elif torch.xpu.is_available():
    print("XPU - available devices:")
    for i in range(torch.xpu.device_count()):
        print(f"  {i}: {torch.xpu.get_device_name(i)}")
    torch.set_default_device('xpu')
else: print("WARNING: Running on CPU")
print(f"Torch version: {torch.__version__}, Device: {torch.get_default_device()}")

XPU - available devices:
  0: Intel(R) Arc(TM) 130V GPU (16GB)
Torch version: 2.10.0+xpu, Device: xpu:0


In [93]:
from huggingface_hub import notebook_login
notebook_login()

In [94]:
import pandas as pd
from collections import OrderedDict

# Mahsa's code for levenshtein distance
def levenshtein(a: str, b: str, case_insensitive=True) -> int:
    # If one of the strings is empty
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)
    
    if case_insensitive:
        a = a.lower()
        b = b.lower()

    # Create distance matrix (size: (len(a)+1) x (len(b)+1))
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]

    # Initialize first row/column
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j

    # Fill in matrix
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost # substitution
            )
    distance = dp[-1][-1]

    return distance

def compare_preds(preds : pd.DataFrame, labels : pd.DataFrame, ignore_trash_columns = False):
    # Drop meta columns that may be included in the preds dataframe
    preds = preds.drop(columns=["fullResponse", "fullAddress", "parseError"], errors="ignore").astype(str)
    labels = labels.astype(str)

    tolerance_levels = 5
    correct_with_tol = [0,] * tolerance_levels
    total_preds = 0
    sum_levenshtein = 0
    sum_similarity = 0.0
    sum_levenshtein_match = 0
    sum_similarity_match = 0.0
    some_match_count = 0
    
    if not ignore_trash_columns:
        # labels that should not have been predicted at all
        trash_predictions = preds[[col for col in preds.columns if col not in labels.columns]].stack()
        total_preds += trash_predictions.notna().sum()
        sum_levenshtein += trash_predictions.dropna().astype(str).str.len().sum()
    for col in labels.columns:
        total_preds += len(labels)
        if col not in preds.columns:
            # all missing predictions are incorrect
            sum_levenshtein += labels[col].dropna().str.len().sum()
        else:
            strings_to_compare = pd.concat([preds[col].fillna(""), labels[col].fillna("")], axis=1)
            levenshtein_scores = strings_to_compare.apply(
                lambda row: levenshtein(row.iloc[0], row.iloc[1]), axis=1
            )
            levenshtein_bounds = strings_to_compare.apply(
                lambda row: max(len(row.iloc[0]), len(row.iloc[1])), axis=1
            )
            similarity = ((levenshtein_bounds - levenshtein_scores) / levenshtein_bounds).fillna(1.0) # nan => div by 0 => both are empty strings => similarity 1.0
            sum_levenshtein += levenshtein_scores.sum()
            sum_similarity += similarity.sum()
            sum_levenshtein_match += levenshtein_scores[similarity >= 0].sum()
            sum_similarity_match += similarity[similarity >= 0].sum()
            some_match_count += (similarity > 0).sum()
            for tol in range(tolerance_levels):
                correct_with_tol[tol] += (levenshtein_scores <= tol).sum()
    results = OrderedDict()
    results["accuracy"] = correct_with_tol[0] / total_preds
    for tol in range(1, tolerance_levels):
        results[f"accuracy_with_tol_{tol}"] = correct_with_tol[tol] / total_preds
    results["average_levenshtein"] = sum_levenshtein / total_preds
    results["average_similarity"] = sum_similarity / total_preds
    results["average_levenshtein_match"] = sum_levenshtein_match / some_match_count if some_match_count > 0 else 0.0
    results["average_similarity_match"] = sum_similarity_match / some_match_count if some_match_count > 0 else 0.0
    results["no_match_rate"] = 1.0 - (some_match_count / total_preds)
    return results

COLS_TO_PREDICT = [
    "HouseNumber",
    "StreetName",
    "City",
    "State",
    "Country"
]

EXCEPT_COUNTRY = COLS_TO_PREDICT[:-1]

In [95]:
import http.client
import json
import urllib.parse
import time

DEFAULT_LIBPOSTAL_LABEL_MAPPING = {
    "house_number": "HouseNumber",
    "road": "StreetName",
    "city": "City",
    "state": "State",
    "country": "Country",
    "postcode": "PostalCode"
}

class LibpostalClient:
    def __init__(self, url: str = "http://localhost:7272", label_mapping = DEFAULT_LIBPOSTAL_LABEL_MAPPING, start_container_if_unavailable: bool = True):
        self.url = url
        parsed_url = urllib.parse.urlparse(url)
        self.host = parsed_url.hostname
        self.port = parsed_url.port
        self.label_mapping = label_mapping or {}
        self.auto_started = False
        if start_container_if_unavailable:
            if not self.start_container_if_needed():
                raise ConnectionError(f"Could not connect to libpostal server at {self.url}, and failed to start docker container.")
    
    def _transform_results(self, parsed_addresses : list[list[list[str]]]) -> list[dict]:
        results = []
        for addr in parsed_addresses:
            result = {}
            for part, label in addr:
                if label in self.label_mapping:
                    label = self.label_mapping[label]
                if label in result:
                    result[label] += "___" + part
                else:
                    result[label] = part
            results.append(result)
        return results
    
    def _health_check(self) -> bool:
        conn = http.client.HTTPConnection(self.host, self.port, timeout=3)
        try:
            conn.request("GET", "/health")
            response = conn.getresponse()
            conn.close()
            return response.status == 204
        except Exception:
            return False
    
    def _start_docker_container(self) -> bool:
        print("Attempting to start libpostal-server docker-compose service...")
        print("This may take a long time on first run since the docker image needs to be built.")
        import subprocess
        result = subprocess.run(["docker-compose", "-f", "docker-compose.yml", "up", "-d", "libpostal-server"], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Failed to start libpostal-server docker container (exit code {result.returncode}):")
            print(result.stdout)
            print(result.stderr)
            return False
        for _ in range(10):
                time.sleep(1)
                if self._health_check():
                    print("Libpostal server is now available.")
                    self.auto_started = True
                    return True
        return False
    
    def start_container_if_needed(self):
        if not self._health_check():
            return self._start_docker_container()
        else: return True
            

    def parse_addresses(self, addresses: list[str]):
        def _request(addresses):
            conn = http.client.HTTPConnection(self.host, self.port, timeout=3)
            headers = {'Content-Type': 'application/json'}
            if isinstance(addresses, str):
                addresses = [addresses]
            elif not isinstance(addresses, list):
                addresses = list(addresses)
            body = json.dumps(addresses)
            
            conn.request("GET", "/parse_addresses", body, headers)
            response = conn.getresponse()
            data = response.read()
            
            results = json.loads(data.decode('utf-8'))
            conn.close()

            return self._transform_results(results)
        try:
            return _request(addresses)
        except Exception as e:
            if self._health_check():
                raise e
            else:
                print(f"Libpostal server not reachable at {self.url}.")
                if not self._start_docker_container():
                    raise e
            return _request(addresses)
    
    def close(self):
        if self.auto_started:
            print("Stopping auto-started libpostal-server docker container...")
            import subprocess
            subprocess.run(["docker-compose", "-f", "docker-compose.yml", "down", "libpostal-server"])

In [96]:
import transformers
from transformers import pipeline

class LlamaAddressSegmentationModel:
    def __init__(self, model_name, prompt, output_parser = None, batch_size=32):
        tokenizer = transformers.AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", padding_side='left')
        self.pipe = pipeline("text-generation", model=model_name, batch_size=batch_size, tokenizer=tokenizer)
        self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id[0]
        self.prompt = prompt
        self.output_parser = output_parser or (lambda x : x)

    def parse_addresses(self, addresses : list[str]) -> str:
        messages = [[
            {"role": "user", "content": self.prompt % {"address" : address}}
        ] for address in addresses]
        result = self.pipe(messages)
        responses = [self.output_parser(r[0]["generated_text"][1]["content"]) for r in result]
        return responses

def extract_json_block(model_response : str):
    limit_chars = [('{', '}'), ('[', ']'), ('"', '"')]
    json_str = model_response
    parts = model_response.split("```")
    if len(parts) >= 2: # single code block or malformed code block
        json_str = parts[1]
    elif len(parts) >= 3:
        for part in parts:
            part = part.strip()
            if part.startswith("json") or any(part.startswith(lim[0]) and part.endswith(lim[1]) for lim in limit_chars):
                json_str = part
                break
    if json_str.startswith("json"):
        json_str = json_str[4:].strip()
    return json_str

In [97]:
import json

class JSONObjectParser:
    def __call__(self, response: str) -> dict:
        try:
            json_str = extract_json_block(response)
            obj = json.loads(json_str)
            for k in obj.keys():
                assert k not in ["fullResponse", "model-fullAddress", "parseError"], f"Key collision with reserved field: {k}"
                # It is plausible that the model just echoes back the full address in a field named "fullAddress"
                # But that would collide with our own field named "fullAddress" that we add later
                if k == "fullAddress":
                    obj["model-fullAddress"] = obj["fullAddress"] # avoid collision but keep the wrongfully generated field
            obj["fullResponse"] = response
            data = obj
        except Exception as e:
            data = {"fullResponse": response, "parseError": str(e)}
        return data

class JSONTuplesParser:
    def __init__(self, ignore_other = True):
        self.ignore_other = ignore_other

    def __call__(self, response: str) -> dict:
        try:
            json_str = extract_json_block(response)
            tuples = json.loads(json_str)
            data = {}
            for part, ptype in tuples:
                if isinstance(part, str) and part.strip() == "":
                    continue
                assert ptype not in ["fullResponse", "model-fullAddress", "parseError"], f"Key collision with reserved field: {ptype}"
                if ptype == "fullAddress":
                    data["model-fullAddress"] = data["fullAddress"] # avoid collision but keep the wrongfully generated field
                else:
                    collision_value = data.get(ptype)
                    if collision_value is None:
                        data[ptype] = part
                    else:
                        data[ptype] = collision_value + "___" + part
            data["fullResponse"] = response
            if self.ignore_other:
                other_key = "Other"
                if isinstance(self.ignore_other, str): other_key = self.ignore_other
                if other_key in data:
                    del data[other_key]
        except Exception as e:
            data = {"fullResponse": response, "parseError": str(e)}
        return data

In [98]:
from collections import OrderedDict
import json
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv")
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_test.csv")

EXAMPLES = [
    ("Berlin, Alexanderplatz 1, 10178", 
     OrderedDict([("City" , "Berlin"), ("StreetName", "Alexanderplatz"), ("HouseNumber", "1")])),
    ("Braunschweig, Uferstr. 25", # From BZK open training set
     OrderedDict([("City", "Braunschweig"), ("StreetName", "Uferstr."), ("HouseNumber", "25")])),
    ("808 Westend Avenue, New York 25, N.Y.", # From BZK open training set
        OrderedDict([("StreetName", "Westend Avenue"), ("HouseNumber", "808"), 
        ("City", "New York"), ("State", "N.Y.")])),
]

PROMPTS = [
    "Segment addresses into their components.\n"
    "Output only a JSON object with the following keys: " + ", ".join(COLS_TO_PREDICT) + ". "
    "Do not include fields that cannot be determined and do not try to guess their values. "
    "For example, if the address is simply \"Berlin\" then the field \"Country\" should be null. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word. "
    "Consider the following examples:\n" +
    ''.join(f"Address: {adr}\n```json\n{json.dumps(ex)}\n```\n" for adr, ex in EXAMPLES) +
    "Now segment the following address:\n%(address)s",

    "Annotate address components with the respective types."
    "Consider the component types: " + ", ".join(COLS_TO_PREDICT + ["Other"]) + ". "
    "Not all addresses will contain all component types and you must not guess the missing ones. "
    "Addresses will most times be written in german, meaning country and city names may be in "
    "german and the addresses "
    "may include german terms such as:\n"
    " - \"burg\" or \"stadt\" for city\n"
    " - \"straße\" or its abbreviation \"str.\" for street.\n"
    "These terms may occur as a suffix to another word.\n"
    "Output only a JSON list of [component, type] tuples. "
    "Consider the following examples:\n" +
    ''.join(f"Address: {adr}\n```json\n{json.dumps([(v,k) for k,v in ex.items()])}\n```\n" for adr, ex in EXAMPLES) +
    "Now annotate the following address:\n%(address)s",
]

In [ ]:

import time
from pathlib import Path

model_configs = [
    {
        "name" : "libpostal",
        "factory": lambda: LibpostalClient(),
        "cleanup": lambda client: client.close(),
    },
    {
        "name" : "Llama-3.2-1B-Instruct-prompt0",
        "factory": lambda: LlamaAddressSegmentationModel(
            "meta-llama/Llama-3.2-1B-Instruct",
            PROMPTS[0],
            JSONObjectParser()
        ),
    },
    {
        "name" : "Llama-3.2-3B-Instruct-prompt0",
        "factory": lambda: LlamaAddressSegmentationModel(
            "meta-llama/Llama-3.2-3B-Instruct",
            PROMPTS[0],
            JSONObjectParser()
        ),
    },
    {
        "name" : "Llama-3.2-1B-Instruct-prompt1",
        "factory": lambda: LlamaAddressSegmentationModel(
            "meta-llama/Llama-3.2-1B-Instruct",
            PROMPTS[1],
            JSONTuplesParser()
        ),
    },
    {
        "name" : "Llama-3.2-3B-Instruct-prompt1",
        "factory": lambda: LlamaAddressSegmentationModel(
            "meta-llama/Llama-3.2-3B-Instruct",
            PROMPTS[1],
            JSONTuplesParser()
        ),
    },
    
]



def eval(dataset, configs=model_configs, pred_cache_path=None):
    if pred_cache_path is not None:
        pred_cache_path = Path(pred_cache_path)
    if pred_cache_path is None or not pred_cache_path.exists():
        cached_preds = {}
    else:
        print(f"Loading cached predictions... To avoid this delete or rename the file {pred_cache_path}")
        with open(pred_cache_path, "r") as f:
            cached_preds = json.load(f)
    
    preds_per_config = []
    model_results = []

    for config in configs:
        config_name = config["name"]
        if config_name in cached_preds:
            print(f"Using cached predictions for model {config_name}...")
            print(f"To avoid this delete or rename the file {pred_cache_path} or delete the entry for {config_name} inside it.")
            preds = cached_preds[config_name]["preds"]
            deltatime = cached_preds[config_name]["deltatime"]
        else:
            print(f"Loading model {config_name}...")
            model = config["factory"]()
            print(f"Segmenting addresses...")
            start = time.monotonic()
            preds = model.parse_addresses(dataset["FullAddress"].tolist())
            deltatime = time.monotonic() - start
            if "cleanup" in config:
                print("Cleaning up model resources...")
                config["cleanup"](model)
            print("Parsing model responses...")
            if pred_cache_path is not None:
                cached_preds[config_name] = {
                    "preds": preds,
                    "deltatime": deltatime
                }
        preds_df = pd.DataFrame(preds)
        preds_per_config.append(preds_df)
        print("Computing metrics...")
        metrics = compare_preds(preds_df, dataset[COLS_TO_PREDICT])
        metrics["deltatime"] = deltatime
        metrics["rate"] = len(dataset) / metrics["deltatime"]
        metrics["parseErrors"] = preds_df["parseError"].notna().sum() if "parseError" in preds_df.columns else 0
        metrics["errorRate"] = metrics["parseErrors"] / len(dataset)
        preds_df["config_name"] = config_name
        preds_df["FullAddress"] = dataset["FullAddress"]

        model_results.append(metrics)
        print(f"Results for model {config_name}: {metrics}")

    if pred_cache_path is not None:
        with open(pred_cache_path, "w") as f:
            json.dump(cached_preds, f)

    preds_per_config_df = pd.concat(preds_per_config)
    default_cols = ["config_name", "FullAddress"] + COLS_TO_PREDICT
    new_order = default_cols + [col for col in preds_per_config_df.columns if col not in default_cols]
    preds_per_config_df = preds_per_config_df[new_order]

    results_df = pd.DataFrame(model_results, index = [config["name"] for config in configs])
    return preds_per_config_df, results_df



preds_per_config, model_statistics = eval(bzkopen_val, model_configs, pred_cache_path=Path("cached_preds_val.json"))

model_statistics

Loading cached predictions... To avoid this delete or rename the file cached_preds.json
Using cached predictions for model libpostal...
To avoid this delete or rename the file cached_preds.json or delete the entry for libpostal inside it.
Computing metrics...
Results for model libpostal: OrderedDict({'accuracy': 0.6907356948228883, 'accuracy_with_tol_1': 0.6961852861035422, 'accuracy_with_tol_2': 0.7016348773841962, 'accuracy_with_tol_3': 0.7070844686648501, 'accuracy_with_tol_4': 0.7316076294277929, 'average_levenshtein': 3.0994550408719346, 'average_similarity': 0.7123332775410771, 'average_levenshtein_match': 2.0974264705882355, 'average_similarity_match': 0.9611261502116738, 'no_match_rate': 0.2588555858310627, 'deltatime': 0.04700000000593718, 'rate': 2808.5106379430927, 'parseErrors': 0, 'errorRate': 0.0})
Using cached predictions for model Llama-3.2-1B-Instruct-prompt0...
To avoid this delete or rename the file cached_preds.json or delete the entry for Llama-3.2-1B-Instruct-prom

,accuracy,accuracy_with_tol_1,accuracy_with_tol_2,accuracy_with_tol_3,accuracy_with_tol_4,average_levenshtein,average_similarity,average_levenshtein_match,average_similarity_match,no_match_rate,deltatime,rate,parseErrors,errorRate
libpostal,0.690736,0.696185,0.701635,0.707084,0.731608,3.099455,0.712333,2.097426,0.961126,0.258856,0.047,2808.510638,0,0.000000
Llama-3.2-1B-Instruct-prompt0,0.531014,0.579425,0.586989,0.602118,0.700454,2.993949,0.553805,5.116580,0.948355,0.416036,123.703,1.067072,1,0.007576
Llama-3.2-3B-Instruct-prompt0,0.782148,0.786687,0.803328,0.821483,0.856278,1.361573,0.805928,1.618018,0.959853,0.160363,267.796,0.492913,12,0.090909
Llama-3.2-1B-Instruct-prompt1,0.766566,0.777108,0.787651,0.805723,0.837349,1.819277,0.801600,2.113677,0.945403,0.152108,146.312,0.902182,20,0.151515
Llama-3.2-3B-Instruct-prompt1,0.827795,0.835347,0.842900,0.862538,0.886707,1.167674,0.851604,1.290155,0.973682,0.125378,313.281,0.421347,8,0.060606


In [100]:

preds_per_config

,config_name,FullAddress,HouseNumber,StreetName,City,State,Country,house,state_district,unit,...,HouseNumber2,StreetType,StreetNumber,parseError,Region,Direction,Suburb,Postal,"HouseNumber, StreetName",City/Other
0,libpostal,"Regensburg, Königstr. 2/I",2/i,königstr.,NaN,NaN,NaN,regensburg,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,libpostal,Dortmund,NaN,NaN,dortmund,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,libpostal,Jöhlingen/Krs. Durlach/Baden.,NaN,NaN,NaN,NaN,NaN,jöhlingen/krs. durlach/baden.,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,libpostal,"8 Burlington Road, Manchester 20/England.",8___20/england.,burlington road manchester,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,libpostal,Leer/Ostfriesland,NaN,NaN,NaN,NaN,NaN,NaN,leer/ostfriesland,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,Llama-3.2-3B-Instruct-prompt1,Sosnowice/Polen,NaN,NaN,Sosnowice,NaN,/Polen,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,Llama-3.2-3B-Instruct-prompt1,2114-79 St. Jackson Heights. N.Y. USA,2114-79,St.,Jackson Heights,N.Y.,USA,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129,Llama-3.2-3B-Instruct-prompt1,Losone CSR,NaN,Losone,NaN,CSR,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
130,Llama-3.2-3B-Instruct-prompt1,Rum.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Expecting value: line 1 column 1 (char 0),NaN,NaN,NaN,NaN,NaN,NaN


In [101]:
default_cols = ["config_name", "FullAddress"] + COLS_TO_PREDICT
preds_vs_trues = preds_per_config[default_cols].merge(
    bzkopen_val[default_cols[1:]], on="FullAddress", suffixes=("_pred", "_true"), how="left")
preds_vs_trues = preds_vs_trues[["config_name", "FullAddress"] + [new_col for col in COLS_TO_PREDICT for new_col in [f"{col}_pred", f"{col}_true"]]] # Order the columns for readability
preds_vs_trues

,config_name,FullAddress,HouseNumber_pred,HouseNumber_true,StreetName_pred,StreetName_true,City_pred,City_true,State_pred,State_true,Country_pred,Country_true
0,libpostal,"Regensburg, Königstr. 2/I",2/i,2/I,königstr.,Königstr.,NaN,Regensburg,NaN,NaN,NaN,NaN
1,libpostal,Dortmund,NaN,NaN,NaN,NaN,dortmund,Dortmund,NaN,NaN,NaN,NaN
2,libpostal,Jöhlingen/Krs. Durlach/Baden.,NaN,NaN,NaN,NaN,NaN,Jöhlingen,NaN,Baden,NaN,NaN
3,libpostal,"8 Burlington Road, Manchester 20/England.",8___20/england.,8,burlington road manchester,Burlington Road,NaN,Manchester,NaN,NaN,NaN,England
4,libpostal,Leer/Ostfriesland,NaN,NaN,NaN,NaN,NaN,Leer,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
715,Llama-3.2-3B-Instruct-prompt1,Sosnowice/Polen,NaN,NaN,NaN,NaN,Sosnowice,Sosnowice,NaN,NaN,/Polen,Polen
716,Llama-3.2-3B-Instruct-prompt1,2114-79 St. Jackson Heights. N.Y. USA,2114-79,NaN,St.,NaN,Jackson Heights,St. Jackson Heights,N.Y.,N.Y.,USA,USA
717,Llama-3.2-3B-Instruct-prompt1,Losone CSR,NaN,NaN,Losone,NaN,NaN,Losone,CSR,NaN,NaN,CSR
718,Llama-3.2-3B-Instruct-prompt1,Rum.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rum.
